# Quantitative Value Strategy
"Value investing" means investing in the stocks that are cheapest relative to common measures of business value (like earnings or assets).

For this project, we're going to build an investing strategy that selects the 50 stocks with the best value metrics. From there, we will calculate recommended trades for an equal-weight portfolio of these 50 stocks.

## Library Imports
The first thing we need to do is import the open-source software libraries that we'll be using in this tutorial.

In [1]:
import numpy as np
import pandas as pd
import xlsxwriter
import requests
from scipy import stats
import math

## Importing Our List of Stocks & API Token
As before, we'll need to import our list of stocks and our API token before proceeding. Make sure the .csv file is still in your working directory and import it with the following command:

In [2]:
stocks = pd.read_csv('sp_500_stocks.csv')
stocks

,Ticker
0,A
1,AAL
2,AAP
3,AAPL
4,ABBV
...,...
500,YUM
501,ZBH
502,ZBRA
503,ZION


## Making Our First API Call
It's now time to make the first version of our value screener!

We'll start by building a simple value screener that ranks securities based on a single metric (the price-to-earnings ratio).

In [3]:
import yfinance as yf

apple = yf.Ticker("AAPL")
print(apple.info)

{'address1': 'One Apple Park Way', 'city': 'Cupertino', 'state': 'CA', 'zip': '95014', 'country': 'United States', 'phone': '(408) 996-1010', 'website': 'https://www.apple.com', 'industry': 'Consumer Electronics', 'industryKey': 'consumer-electronics', 'industryDisp': 'Consumer Electronics', 'sector': 'Technology', 'sectorKey': 'technology', 'sectorDisp': 'Technology', 'longBusinessSummary': 'Apple Inc. designs, manufactures, and markets smartphones, personal computers, tablets, wearables, and accessories worldwide. The company offers iPhone, a line of smartphones; Mac, a line of personal computers; iPad, a line of multi-purpose tablets; and wearables, home, and accessories comprising AirPods, Apple Vision Pro, Apple TV, Apple Watch, Beats products, and HomePod, as well as Apple branded and third-party accessories. It also provides AppleCare support and cloud services; and operates various platforms, including the App Store that allow customers to discover and download applications and

In [5]:
tickers = stocks["Ticker"].dropna().tolist()
tickers = [ticker.strip() for ticker in tickers]

data = []

for ticker in tickers:
    try:
        stock = yf.Ticker(ticker)
        info = stock.info

        # P/E Ratio: Price / EPS -> trailingPe
        # trailingPE: historisches KGV (TTM = Trailing Twelve Months)
        # forwardPE : erwartetes KGV (Forward)
        # trailingEps : Gewinn pro Aktie (EPS)
        price = info.get("currentPrice")
        trailing_eps = info.get("trailingEps")
        trailing_pe = info.get("trailingPE")
        forward_pe = info.get("forwardPE")

        if trailing_pe is None and price is not None and trailing_eps not in [None, 0]:
            trailing_pe = price / trailing_eps
        
        data.append({
            "Ticker": ticker,
            "Price": info.get("currentPrice"),
            "Trailing EPS" : trailing_eps,
            "Trailing PE" : trailing_pe, 
            "Forawr PE" : forward_pe,
            "MarketCap": info.get("marketCap")
        })
    except Exception as e:
        data.append({
            "Ticker": ticker,
            "Price": np.nan,
            "Trailing EPS": np.nan,
            "Trailing P/E": np.nan,
            "Forward P/E": np.nan
        })

stocksDf = pd.DataFrame(data)
# readable number
stocksDf ["MarketCap_Billion"] = stocksDf["MarketCap"] / 1e9

stocksDf

HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: ANSS"}}}
HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: CMA"}}}
HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: HBI"}}}
HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: HES"}}}
HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: IPG"}}}
HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: K"}}}
HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: WBA"}}}


,Ticker,Price,Trailing EPS,Trailing PE,Forawr PE,MarketCap,MarketCap_Billion
0,A,118.755,4.52,26.273230,18.042004,3.356044e+10,33.560437
1,AAL,12.855,0.31,41.467740,5.839678,8.502106e+09,8.502106
2,AAP,58.540,1.13,51.805310,14.892151,3.531672e+09,3.531672
3,AAPL,285.840,8.27,34.557434,29.859590,4.197500e+12,4197.499535
4,ABBV,204.730,2.04,100.357840,12.633505,3.621187e+11,362.118709
...,...,...,...,...,...,...,...
500,YUM,156.690,6.20,25.272581,20.900974,4.331384e+10,43.313840
501,ZBH,83.740,3.86,21.694300,9.311480,1.620055e+10,16.200547
502,ZBRA,229.320,8.18,28.034230,11.436699,1.128064e+10,11.280642
503,ZION,64.410,6.44,10.001554,9.652084,9.473230e+09,9.473230


## Removing Glamour Stocks

The opposite of a "value stock" is a "glamour stock". 

Since the goal of this strategy is to identify the 50 best value stocks from our universe, our next step is to remove glamour stocks from the DataFrame.

We'll sort the DataFrame by the stocks' price-to-earnings ratio, and drop all stocks outside the top 50.

In [6]:
final_stocks_df = stocksDf.sort_values("Trailing PE", ascending=False)
final_stocks_df = final_stocks_df[:50]
final_stocks_df.reset_index(inplace = True)
final_stocks_df

,index,Ticker,Price,Trailing EPS,Trailing PE,Forawr PE,MarketCap,MarketCap_Billion
0,206,GPC,106.2100,0.44,241.386370,12.684698,1.477616e+10,14.776159
1,175,FANG,195.6299,0.99,197.605960,11.329318,5.503299e+10,55.032988
2,384,PVH,91.4000,0.52,175.769240,6.658944,4.187174e+09,4.187174
3,31,AMD,415.0050,2.59,160.220080,33.760235,6.765716e+11,676.571578
4,474,VTR,87.1500,0.55,158.454540,96.120970,4.236969e+10,42.369692
5,249,IRM,133.4450,0.92,145.048920,49.910050,3.970318e+10,39.703183
6,246,IPGP,98.5800,0.68,144.970600,37.811546,4.184068e+09,4.184068
7,127,DD,49.8300,0.38,131.131590,19.135944,2.042418e+10,20.424182
8,385,PWR,781.4650,7.25,107.788280,48.107327,1.172668e+11,117.266751
9,481,WELL,215.2100,2.08,103.466354,65.149414,1.519199e+11,151.919854


## Building a Better (and More Realistic) Value Strategy
Every valuation metric has certain flaws.

For example, the price-to-earnings ratio doesn't work well with stocks with negative earnings.

Similarly, stocks that buyback their own shares are difficult to value using the price-to-book ratio.

Investors typically use a `composite` basket of valuation metrics to build robust quantitative value strategies. In this section, we will filter for stocks with the lowest percentiles on the following metrics:

* Price-to-earnings ratio
* Price-to-book ratio
* Price-to-sales ratio
* Enterprise Value divided by Earnings Before Interest, Taxes, Depreciation, and Amortization (EV/EBITDA)
* Enterprise Value divided by Gross Profit (EV/GP)

Some of these metrics aren't provided directly by the IEX Cloud API, and must be computed after pulling raw data. We'll start by calculating each data point from scratch.

In [8]:
from tqdm import tqdm

tickers = stocks["Ticker"].dropna().tolist()
tickers = [ticker.strip() for ticker in tickers]

rv_columns = [
    "Ticker",
    "Price",
    "Number of Shares To Buy",
    "Price-to-Earnings Ratio",
    "PE Percentile",
    "Price-to-Book Ratio",
    "PB Percentile",
    "Price-to-Sales Ratio",
    "PS Percentile",
    "EV/EBITDA",
    "EV/EBITDA Percentile",
    "EV/GP",
    "EV/GP Percentile",
]

rows = []

for ticker in tqdm(tickers):
    try:
        stock = yf.Ticker(ticker)
        info = stock.info

        price = info.get("currentPrice")
        pe_ratio = info.get("trailingPE")
        pb_ratio = info.get("priceToBook")
        ps_ratio = info.get("priceToSalesTrailing12Months")
        ev_to_ebitda = info.get("enterpriseToEbitda")
        enterprise_value = info.get("enterpriseValue")
        gross_profit = info.get("grossProfits")
        ev_to_gp = np.nan
        
        if trailing_pe is None and price is not None and trailing_eps not in [None, 0]:
            trailing_pe = price / trailing_eps
        if enterprise_value not in [None, 0] and gross_profit not in [None, 0]:
            ev_to_gp = enterprise_value / gross_profit
        
        rows.append({
            "Ticker": ticker,
            "Price": price,
            "Number of Shares To Buy": 0,
            "Price-to-Earnings Ratio": pe_ratio,
            "PE Percentile": np.nan,
            "Price-to-Book Ratio": pb_ratio,
            "PB Percentile": np.nan,
            "Price-to-Sales Ratio": ps_ratio,
            "PS Percentile": np.nan,
            "EV/EBITDA": ev_to_ebitda,
            "EV/EBITDA Percentile": np.nan,
            "EV/GP": ev_to_gp,
            "EV/GP Percentile": np.nan,
        })

    except Exception as e:
        rows.append({
            "Ticker": ticker,
            "Price": np.nan,
            "Number of Shares To Buy": 0,
            "Price-to-Earnings Ratio": np.nan,
            "PE Percentile": np.nan,
            "Price-to-Book Ratio": np.nan,
            "PB Percentile": np.nan,
            "Price-to-Sales Ratio": np.nan,
            "PS Percentile": np.nan,
            "EV/EBITDA": np.nan,
            "EV/EBITDA Percentile": np.nan,
            "EV/GP": np.nan,
            "EV/GP Percentile": np.nan,
        })

rv_df = pd.DataFrame(rows, columns=rv_columns)
# readable number

rv_df

100%|██████████| 505/505 [04:18<00:00,  1.95it/s]


,Ticker,Price,Number of Shares To Buy,Price-to-Earnings Ratio,PE Percentile,Price-to-Book Ratio,PB Percentile,Price-to-Sales Ratio,PS Percentile,EV/EBITDA,EV/EBITDA Percentile,EV/GP,EV/GP Percentile
0,A,118.3400,0,26.181416,NaN,4.842854,NaN,4.733639,NaN,17.761,NaN,9.473951,NaN
1,AAL,12.8201,0,41.355160,NaN,-2.079835,NaN,0.151427,NaN,8.593,NaN,2.745478,NaN
2,AAP,58.4350,0,51.712390,NaN,1.595147,NaN,0.409875,NaN,12.591,NaN,1.578503,NaN
3,AAPL,285.2800,0,34.495766,NaN,47.562520,NaN,9.281388,NaN,26.192,NaN,19.392037,NaN
4,ABBV,204.6600,0,100.323530,NaN,-110.627030,NaN,5.762507,NaN,14.253,NaN,9.451052,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
500,YUM,156.4750,0,25.237906,NaN,-5.917222,NaN,5.097149,NaN,17.979,NaN,13.868036,NaN
501,ZBH,83.4550,0,21.620468,NaN,1.273558,NaN,1.919993,NaN,8.979,NaN,3.927340,NaN
502,ZBRA,228.8275,0,27.974020,NaN,3.162829,NaN,2.086066,NaN,14.017,NaN,5.324679,NaN
503,ZION,64.4700,0,10.010870,NaN,1.338079,NaN,2.795417,NaN,NaN,NaN,2.412432,NaN


## Calculating Value Percentiles

We now need to calculate value score percentiles for every stock in the universe. More specifically, we need to calculate percentile scores for the following metrics for every stock:

* Price-to-earnings ratio
* Price-to-book ratio
* Price-to-sales ratio
* EV/EBITDA
* EV/GP

Here's how we'll do this:

In [ ]:
metrics = {
    "Price-to-Earnings Ratio": "PE Percentile",
    "Price-to-Book Ratio": "PB Percentile",
    "Price-to-Sales Ratio": "PS Percentile",
    "EV/EBITDA": "EV/EBITDA Percentile",
    "EV/GP": "EV/GP Percentile",
}

for metric, percentile_col in metrics.items():
    rv_df[metric] = pd.to_numeric(rv_df[metric], errors="coerce")
    
    # pct: Percentile
    # pct=True -> instead of a rank it returns a value between 0 and 1
    #   e.g.: 0.95 -> better than 95% of the stocks
    rv_df[percentile_col] = rv_df[metric].rank(
        pct=True,
        ascending=False
    )

rv_df

,Ticker,Price,Number of Shares To Buy,Price-to-Earnings Ratio,PE Percentile,Price-to-Book Ratio,PB Percentile,Price-to-Sales Ratio,PS Percentile,EV/EBITDA,EV/EBITDA Percentile,EV/GP,EV/GP Percentile
0,A,118.3400,0,26.181416,0.444976,4.842854,0.322940,4.733639,0.285714,17.761,0.342043,9.473951,0.420225
1,AAL,12.8201,0,41.355160,0.160287,-2.079835,0.933185,0.151427,0.993304,8.593,0.843230,2.745478,0.921348
2,AAP,58.4350,0,51.712390,0.100478,1.595147,0.772829,0.409875,0.946429,12.591,0.631829,1.578503,0.973034
3,AAPL,285.2800,0,34.495766,0.232057,47.562520,0.017817,9.281388,0.102679,26.192,0.125891,19.392037,0.085393
4,ABBV,204.6600,0,100.323530,0.026316,-110.627030,0.993318,5.762507,0.234375,14.253,0.517815,9.451052,0.422472
...,...,...,...,...,...,...,...,...,...,...,...,...,...
500,YUM,156.4750,0,25.237906,0.464115,-5.917222,0.944321,5.097149,0.258929,17.979,0.332542,13.868036,0.231461
501,ZBH,83.4550,0,21.620468,0.607656,1.273558,0.837416,1.919993,0.660714,8.979,0.826603,3.927340,0.831461
502,ZBRA,228.8275,0,27.974020,0.399522,3.162829,0.461024,2.086066,0.625000,14.017,0.534442,5.324679,0.716854
503,ZION,64.4700,0,10.010870,0.940191,1.338079,0.826281,2.795417,0.549107,NaN,NaN,2.412432,0.937079


## Calculating the RV Score
We'll now calculate our RV Score (which stands for Robust Value), which is the value score that we'll use to filter for stocks in this investing strategy.

The RV Score will be the arithmetic mean of the 4 percentile scores that we calculated in the last section.

To calculate arithmetic mean, we will use the mean function from Python's built-in statistics module.

In [11]:
from statistics import mean

percentile_cols = [
    "PE Percentile",
    "PB Percentile",
    "PS Percentile",
    "EV/EBITDA Percentile",
    "EV/GP Percentile",
]

rv_df["RV Score"] = rv_df[percentile_cols].mean(axis=1)

rv_df.sort_values("RV Score", ascending=False, inplace=True)

rv_df

,Ticker,Price,Number of Shares To Buy,Price-to-Earnings Ratio,PE Percentile,Price-to-Book Ratio,PB Percentile,Price-to-Sales Ratio,PS Percentile,EV/EBITDA,EV/EBITDA Percentile,EV/GP,EV/GP Percentile,RV Score
149,DXC,11.610,0,5.047826,0.995215,0.627703,0.919822,0.155398,0.991071,2.546,0.995249,1.584315,0.970787,0.974429
272,KSS,14.365,0,6.035714,0.990431,0.397449,0.930958,0.103852,0.997768,6.244,0.952494,1.199406,0.986517,0.971633
285,LNC,37.440,0,6.421955,0.988038,0.717296,0.910913,0.391860,0.953125,-24.700,0.997625,-6.754126,1.000000,0.969940
226,HPQ,21.405,0,8.107954,0.968900,-25.634731,0.975501,0.349454,0.962054,6.086,0.957245,2.382255,0.941573,0.961054
111,COTY,2.415,0,NaN,NaN,0.600896,0.924276,0.365956,0.959821,6.833,0.928741,1.486592,0.979775,0.948153
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
467,VIAC,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
478,WBA,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
484,WLTW,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
489,WRK,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Selecting the 50 Best Value Stocks¶

As before, we can identify the 50 best value stocks in our universe by sorting the DataFrame on the RV Score column and dropping all but the top 50 entries.

In [12]:
final_rv_df = rv_df.sort_values("RV Score", ascending=False)
final_rv_df = final_rv_df[:50]
final_rv_df.reset_index(inplace = True)
final_rv_df

,index,Ticker,Price,Number of Shares To Buy,Price-to-Earnings Ratio,PE Percentile,Price-to-Book Ratio,PB Percentile,Price-to-Sales Ratio,PS Percentile,EV/EBITDA,EV/EBITDA Percentile,EV/GP,EV/GP Percentile,RV Score
0,149,DXC,11.6100,0,5.047826,0.995215,0.627703,0.919822,0.155398,0.991071,2.546,0.995249,1.584315,0.970787,0.974429
1,272,KSS,14.3650,0,6.035714,0.990431,0.397449,0.930958,0.103852,0.997768,6.244,0.952494,1.199406,0.986517,0.971633
2,285,LNC,37.4400,0,6.421955,0.988038,0.717296,0.910913,0.391860,0.953125,-24.700,0.997625,-6.754126,1.000000,0.969940
3,226,HPQ,21.4050,0,8.107954,0.968900,-25.634731,0.975501,0.349454,0.962054,6.086,0.957245,2.382255,0.941573,0.961054
4,111,COTY,2.4150,0,NaN,NaN,0.600896,0.924276,0.365956,0.959821,6.833,0.928741,1.486592,0.979775,0.948153
5,104,CNC,54.0600,0,NaN,NaN,1.245766,0.844098,0.149691,0.995536,6.503,0.940618,1.003259,0.991011,0.942816
6,99,CMCSA,26.5050,0,5.197059,0.992823,1.073990,0.879733,0.755780,0.859375,5.087,0.985748,2.048000,0.959551,0.935446
7,456,UHS,170.0900,0,7.101879,0.978469,1.391523,0.815145,0.584481,0.906250,5.715,0.966746,1.940902,0.961798,0.925681
8,93,CHTR,158.3450,0,4.284226,1.000000,1.188527,0.855234,0.409160,0.948661,5.481,0.973872,3.983261,0.826966,0.920947
9,380,PRGO,12.1800,0,NaN,NaN,0.570920,0.926503,0.394199,0.950893,7.240,0.907363,3.292632,0.887640,0.918100
